# 现代时间序列预测

用滞后特征和梯度提升预测日温度。

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "tutorials" / "common.py").exists():
        project_root = str(candidate)
        if project_root not in sys.path:
            sys.path.insert(0, project_root)
        break
else:
    raise FileNotFoundError("未找到项目根目录")

from tutorials.common import DATA_DIR, RANDOM_STATE, display, regression_metrics

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit


## 1. 读取数据

最后 30 天作为测试集。

In [ ]:
temperature = (
    pl.read_csv(DATA_DIR / "temperature.csv")
    .with_columns(pl.col("Date").str.to_date())
    .sort("Date")
)

fig, ax = plt.subplots(figsize=(10, 4), dpi=120)
ax.plot(temperature["Date"].to_list(), temperature["Temp"].to_numpy(), linewidth=1)
ax.set_title("日温度序列")
ax.set_ylabel("温度")
ax.grid(alpha=0.25)
plt.show()

display(temperature.head())
display(temperature.tail())


## 2. 特征工程

构造滞后、滚动和周期特征。

In [ ]:
def make_features(frame):
    day_of_year = pl.col("Date").dt.ordinal_day()
    return (
        frame.with_columns(
            [pl.col("Temp").shift(lag).alias(f"lag_{lag}") for lag in [1, 2, 3, 7, 14, 30]]
            + [
                pl.col("Temp").shift(1).rolling_mean(7).alias("rolling_mean_7"),
                pl.col("Temp").shift(1).rolling_std(7).alias("rolling_std_7"),
                pl.col("Temp").shift(1).rolling_mean(30).alias("rolling_mean_30"),
                pl.col("Temp").shift(1).rolling_std(30).alias("rolling_std_30"),
                day_of_year.alias("day_of_year"),
                (2 * np.pi * day_of_year / 365.25).sin().alias("season_sin"),
                (2 * np.pi * day_of_year / 365.25).cos().alias("season_cos"),
            ]
        )
        .drop_nulls()
    )

featured = make_features(temperature)
feature_columns = [column for column in featured.columns if column not in {"Date", "Temp"}]
train = featured.head(featured.height - 30)
test = featured.tail(30)

X_train = train.select(feature_columns).to_numpy()
y_train = train["Temp"].to_numpy()
X_test = test.select(feature_columns).to_numpy()
y_test = test["Temp"].to_numpy()

display(featured.select(["Date", "Temp", *feature_columns]).head())
print(f"训练样本: {len(train)}，测试样本: {len(test)}，特征数: {len(feature_columns)}")


## 3. 时序验证

用 TimeSeriesSplit 保持时间顺序。

In [ ]:
def build_model():
    return HistGradientBoostingRegressor(
        max_iter=400,
        learning_rate=0.05,
        l2_regularization=0.01,
        random_state=RANDOM_STATE,
    )


records = []
splitter = TimeSeriesSplit(n_splits=5, test_size=30)
for fold, (train_index, valid_index) in enumerate(splitter.split(X_train), start=1):
    model = build_model()
    model.fit(X_train[train_index], y_train[train_index])
    prediction = model.predict(X_train[valid_index])
    records.append(
        {
            "fold": fold,
            "train_end": train["Date"][int(train_index[-1])],
            "valid_start": train["Date"][int(valid_index[0])],
            "valid_end": train["Date"][int(valid_index[-1])],
            "MAE": mean_absolute_error(y_train[valid_index], prediction),
            "RMSE": root_mean_squared_error(y_train[valid_index], prediction),
        }
    )

cv_report = pl.DataFrame(records)
display(cv_report)
display(cv_report.select(pl.mean("MAE").alias("avg_MAE"), pl.mean("RMSE").alias("avg_RMSE")))


## 4. 训练评估

训练最终模型并评估测试集。

In [ ]:
model = build_model()
model.fit(X_train, y_train)
forecast = model.predict(X_test)
metrics = regression_metrics(y_test, forecast)

prediction = pl.DataFrame(
    {
        "Date": test["Date"],
        "actual": y_test,
        "forecast": forecast,
        "error": y_test - forecast,
    }
)

fig, ax = plt.subplots(figsize=(10, 4), dpi=120)
recent = temperature.filter(pl.col("Date") >= test["Date"][0].replace(year=test["Date"][0].year - 1))
ax.plot(recent["Date"].to_list(), recent["Temp"].to_numpy(), label="实际值", linewidth=1)
ax.plot(test["Date"].to_list(), forecast, label="预测值", linewidth=2)
ax.axvline(test["Date"][0], color="gray", linestyle="--", alpha=0.8)
ax.set_title("梯度提升日温度预测")
ax.set_ylabel("温度")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

display(prediction)
display(metrics.round(4))


## 5. 特征重要性

用置换重要性解释模型。

In [ ]:
importance = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=20,
    random_state=RANDOM_STATE,
    scoring="neg_root_mean_squared_error",
)
importance_report = (
    pl.DataFrame(
        {
            "feature": feature_columns,
            "importance_mean": importance.importances_mean,
            "importance_std": importance.importances_std,
        }
    )
    .sort("importance_mean", descending=True)
)

display(importance_report)
